# Titanic Dataset: Complete Exploratory Data Analysis (EDA)

This notebook provides a comprehensive exploratory data analysis of the Titanic dataset, including data loading, cleaning, visualization, and statistical insights.

## Configuration & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import chi2_contingency, pointbiserialr
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_theme(style='whitegrid')
%matplotlib inline

# Color palette
colors = {'survived': '#2ecc71', 'died': '#e74c3c'}
palette_survival = {0: colors['died'], 1: colors['survived']}

print("✓ Libraries loaded successfully")

## 1. Data Loading & Overview

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('/content/Titanic-Dataset.csv')
except FileNotFoundError:
    try:
        df = pd.read_csv('Titanic-Dataset.csv')  # Local path fallback
    except FileNotFoundError:
        print('❌ Error: Titanic-Dataset.csv not found. Please upload or specify correct path.')
        raise

# Create a copy for cleaning (preserve original)
df_original = df.copy()

print("Dataset Overview")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print("\n--- First 5 Rows ---")
display(df.head())
print("\n--- Data Types & Info ---")
df.info()
print("\n--- Summary Statistics (Numerical) ---")
display(df.describe())

## 2. Missing Values & Data Quality

In [ ]:
# Analyze missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)

missing_data = missing_data[missing_data['Missing_Count'] > 0]

print("Missing Values Analysis:")
display(missing_data)
print(f"\n✓ Duplicate rows: {df.duplicated().sum()}")

# Visualization
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='RdYlGn_r')
plt.title("Missing Data Heatmap", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Preprocessing

In [ ]:
print("Data Cleaning Strategy:")
print("="*50)

# Age: Fill with median by Pclass (age varies by ticket class)
print("1. Age: Impute with median by Passenger Class")
df['Age'].fillna(df.groupby('Pclass')['Age'].transform('median'), inplace=True)
print(f"   ✓ Age missing values: {df['Age'].isnull().sum()}")

# Embarked: Fill with mode (most common port)
print("\n2. Embarked: Fill with mode")
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
print(f"   ✓ Embarked missing values: {df['Embarked'].isnull().sum()}")

# Fare: Fill with median by Pclass
print("\n3. Fare: Impute with median by Passenger Class")
df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'), inplace=True)
print(f"   ✓ Fare missing values: {df['Fare'].isnull().sum()}")

# Cabin: Drop (too many missing values - 77%)
print("\n4. Cabin: Drop (77% missing)")
df.drop('Cabin', axis=1, inplace=True)
print(f"   ✓ Cabin dropped")

print(f"\n{'='*50}")
print(f"Cleaned Dataset Shape: {df.shape}")
print(f"Remaining Missing Values: {df.isnull().sum().sum()}")

## 4. Feature Engineering

In [ ]:
# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
print(f"✓ Created 'Title' feature")
print(f"  Unique titles: {df['Title'].unique()}")

# Simplify rare titles
title_mapping = {
    'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master',
    'Dr': 'Dr', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mille': 'Mrs', 'Mme': 'Mrs', 'Ms': 'Mrs', 'Prince': 'Rare',
    'Sir': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Dona': 'Rare', 'Don': 'Rare'
}
df['Title'] = df['Title'].map(title_mapping)
print(f"  Simplified titles: {df['Title'].unique()}")

# Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
print(f"\n✓ Created 'FamilySize' feature (range: {df['FamilySize'].min()}-{df['FamilySize'].max()})")

# Is Alone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
print(f"✓ Created 'IsAlone' feature")

# Age Groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], 
                         labels=['Child', 'Teen', 'Adult', 'Senior', 'Elderly'])
print(f"✓ Created 'AgeGroup' feature")

# Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
print(f"✓ Created 'FarePerPerson' feature")

print(f"\nNew features added: {[col for col in df.columns if col not in df_original.columns]}")

## 5. Numerical Data Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age Distribution
sns.histplot(df['Age'], kde=True, ax=axes[0, 0], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Age Distribution', fontsize=12, fontweight='bold')
axes[0, 0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Mean: {df["Age"].mean():.1f}')
axes[0, 0].legend()

# Age Boxplot
sns.boxplot(y=df['Age'], ax=axes[0, 1], color='skyblue')
axes[0, 1].set_title('Age Boxplot (Outlier Detection)', fontsize=12, fontweight='bold')

# Fare Distribution
sns.histplot(df['Fare'], kde=True, ax=axes[1, 0], color='salmon', edgecolor='black')
axes[1, 0].set_title('Fare Distribution', fontsize=12, fontweight='bold')
axes[1, 0].axvline(df['Fare'].mean(), color='red', linestyle='--', label=f'Mean: {df["Fare"].mean():.2f}')
axes[1, 0].legend()

# Fare Boxplot
sns.boxplot(y=df['Fare'], ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Fare Boxplot (Outlier Detection)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistics
print("Numerical Features Statistics:")
print(f"\nAge - Mean: {df['Age'].mean():.2f}, Median: {df['Age'].median():.2f}, Std: {df['Age'].std():.2f}")
print(f"Fare - Mean: {df['Fare'].mean():.2f}, Median: {df['Fare'].median():.2f}, Std: {df['Fare'].std():.2f}")

## 6. Categorical Analysis & Survival Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Survival by Gender
gender_survival = df.groupby('Sex')['Survived'].agg(['sum', 'count'])
gender_survival['rate'] = (gender_survival['sum'] / gender_survival['count'] * 100).round(1)
sns.histplot(data=df, x='Sex', hue='Survived', ax=axes[0, 0], palette=palette_survival, 
             discrete=True, stat='count', shrink=0.8)
axes[0, 0].set_title('Survival by Gender', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Count')
for i, sex in enumerate(['female', 'male']):
    if sex in gender_survival.index:
        rate = gender_survival.loc[sex, 'rate']
        axes[0, 0].text(i, gender_survival.loc[sex, 'count'] + 10, f'{rate}%', 
                        ha='center', fontweight='bold')

# Survival by Passenger Class
class_survival = df.groupby('Pclass')['Survived'].agg(['sum', 'count'])
class_survival['rate'] = (class_survival['sum'] / class_survival['count'] * 100).round(1)
sns.histplot(data=df, x='Pclass', hue='Survived', ax=axes[0, 1], palette=palette_survival,
             discrete=True, stat='count', shrink=0.8)
axes[0, 1].set_title('Survival by Passenger Class', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Count')
for i in [1, 2, 3]:
    if i in class_survival.index:
        rate = class_survival.loc[i, 'rate']
        axes[0, 1].text(i, class_survival.loc[i, 'count'] + 10, f'{rate}%', 
                        ha='center', fontweight='bold')

# Survival by Port of Embarkation
port_survival = df.groupby('Embarked')['Survived'].agg(['sum', 'count'])
port_survival['rate'] = (port_survival['sum'] / port_survival['count'] * 100).round(1)
sns.histplot(data=df, x='Embarked', hue='Survived', ax=axes[1, 0], palette=palette_survival,
             discrete=True, stat='count', shrink=0.8)
axes[1, 0].set_title('Survival by Port of Embarkation', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Count')

# Survival by Title
title_survival = df.groupby('Title')['Survived'].agg(['sum', 'count'])
title_survival['rate'] = (title_survival['sum'] / title_survival['count'] * 100).round(1)
sns.histplot(data=df, x='Title', hue='Survived', ax=axes[1, 1], palette=palette_survival,
             discrete=True, stat='count', shrink=0.8)
axes[1, 1].set_title('Survival by Title', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 7. Statistical Testing

In [ ]:
print("Statistical Significance Tests")
print("="*60)

# Chi-square test for categorical variables
categorical_vars = ['Sex', 'Pclass', 'Embarked', 'IsAlone']
print("\n1. Chi-Square Test (Categorical vs Survival)")
print("-" * 60)
for var in categorical_vars:
    contingency_table = pd.crosstab(df[var], df['Survived'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    significance = "✓ Significant" if p_value < 0.05 else "✗ Not Significant"
    print(f"{var:15} - Chi2: {chi2:.4f}, p-value: {p_value:.4e} [{significance}]")

# Point-biserial correlation for numerical variables
print("\n2. Point-Biserial Correlation (Numerical vs Survival)")
print("-" * 60)
numerical_vars = ['Age', 'Fare', 'FamilySize', 'FarePerPerson']
for var in numerical_vars:
    corr, p_value = pointbiserialr(df['Survived'], df[var])
    significance = "✓ Significant" if p_value < 0.05 else "✗ Not Significant"
    print(f"{var:15} - Correlation: {corr:.4f}, p-value: {p_value:.4e} [{significance}]")

print("\n" + "="*60)

## 8. Correlation Analysis

In [ ]:
# Select only numerical columns for correlation
numerical_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', 
            cbar_kws={'label': 'Correlation Coefficient'}, 
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap - Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Highlight strong correlations with Survived
print("\nTop Correlations with Survival:")
survival_corr = corr_matrix['Survived'].sort_values(ascending=False)
print(survival_corr[survival_corr.index != 'Survived'])

## 9. Outlier Detection (IQR Method)

In [ ]:
def calculate_outliers_iqr(series, column_name):
    """Calculate outliers using IQR method"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_mask = (series < lower_bound) | (series > upper_bound)
    outlier_count = outliers_mask.sum()
    outlier_percent = (outlier_count / len(series) * 100)
    
    return {
        'column': column_name,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'count': outlier_count,
        'percent': outlier_percent
    }

print("Outlier Detection Using IQR Method")
print("="*60)

outlier_results = []
for col in ['Age', 'Fare', 'FamilySize']:
    result = calculate_outliers_iqr(df[col].dropna(), col)
    outlier_results.append(result)
    print(f"\n{col}:")
    print(f"  Lower Bound: {result['lower_bound']:.2f}")
    print(f"  Upper Bound: {result['upper_bound']:.2f}")
    print(f"  Outliers: {result['count']} ({result['percent']:.2f}%)")

print("\n" + "="*60)
print("Note: Outliers are identified but retained for analysis.")

## 10. Interactive Visualizations

In [ ]:
# Interactive Scatter Plot: Age vs Fare
fig = px.scatter(df, x='Age', y='Fare', color='Survived',
                 hover_data=['Name', 'Sex', 'Pclass', 'Title'],
                 title='Interactive: Age vs Fare (colored by Survival)',
                 color_continuous_scale=['#e74c3c', '#2ecc71'],
                 labels={'Survived': 'Survived'},
                 height=600)
fig.show()

print("Tip: Hover over points to see passenger details")

## 11. Age vs Survival Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Age distribution by survival
df[df['Survived'] == 0]['Age'].hist(bins=20, ax=axes[0], alpha=0.6, label='Died', color='#e74c3c')
df[df['Survived'] == 1]['Age'].hist(bins=20, ax=axes[0], alpha=0.6, label='Survived', color='#2ecc71')
axes[0].set_xlabel('Age', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Age Distribution by Survival Status', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Age group survival rate
agegroup_survival = df.groupby('AgeGroup')['Survived'].agg(['sum', 'count'])
agegroup_survival['rate'] = (agegroup_survival['sum'] / agegroup_survival['count'] * 100)
agegroup_survival['rate'].plot(kind='bar', ax=axes[1], color='#3498db', edgecolor='black')
axes[1].set_xlabel('Age Group', fontsize=11)
axes[1].set_ylabel('Survival Rate (%)', fontsize=11)
axes[1].set_title('Survival Rate by Age Group', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50% baseline')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nSurvival Rate by Age Group:")
print(agegroup_survival[['count', 'sum', 'rate']].rename(columns={'sum': 'Survived', 'count': 'Total'}))

## 12. Key Findings & Insights

In [ ]:
print("\n" + "="*70)
print(" " * 15 + "KEY FINDINGS & INSIGHTS")
print("="*70)

# Overall statistics
total_passengers = len(df)
survived = df['Survived'].sum()
mortality_rate = (1 - survived/total_passengers) * 100

print(f"\n📊 OVERALL STATISTICS")
print("-" * 70)
print(f"  Total Passengers: {total_passengers:,}")
print(f"  Survived: {survived:,} ({survived/total_passengers*100:.1f}%)")
print(f"  Perished: {total_passengers - survived:,} ({mortality_rate:.1f}%)")

# Gender analysis
print(f"\n👥 GENDER ANALYSIS")
print("-" * 70)
for sex in ['female', 'male']:
    sex_data = df[df['Sex'] == sex]
    sex_survived = sex_data['Survived'].sum()
    sex_rate = (sex_survived / len(sex_data)) * 100
    print(f"  {sex.capitalize():8} - Survived: {sex_survived:3}/{len(sex_data):3} ({sex_rate:.1f}%)")
print("  💡 Women had significantly higher survival rates (Women First policy)")

# Class analysis
print(f"\n💎 CLASS ANALYSIS")
print("-" * 70)
for pclass in [1, 2, 3]:
    class_data = df[df['Pclass'] == pclass]
    class_survived = class_data['Survived'].sum()
    class_rate = (class_survived / len(class_data)) * 100
    print(f"  Class {pclass} - Survived: {class_survived:3}/{len(class_data):3} ({class_rate:.1f}%)")
print("  💡 First class passengers had much better survival chances")

# Age analysis
print(f"\n🎂 AGE ANALYSIS")
print("-" * 70)
children = df[df['Age'] < 13]
children_survived = children['Survived'].sum()
print(f"  Children (<13): Survived {children_survived}/{len(children)} ({children_survived/len(children)*100:.1f}%)")
print(f"  💡 Children had higher survival rates (Children First policy)")

# Family size analysis
print(f"\n👨‍👩‍👧‍👦 FAMILY SIZE ANALYSIS")
print("-" * 70)
alone = df[df['IsAlone'] == 1]
with_family = df[df['IsAlone'] == 0]
print(f"  Traveling Alone: {alone['Survived'].sum()}/{len(alone)} survived ({alone['Survived'].sum()/len(alone)*100:.1f}%)")
print(f"  With Family: {with_family['Survived'].sum()}/{len(with_family)} survived ({with_family['Survived'].sum()/len(with_family)*100:.1f}%)")
print(f"  💡 Traveling with family improved survival chances")

print("\n" + "="*70)

## 13. Summary

In [ ]:
print("\n" + "="*70)
print(" " * 20 + "ANALYSIS SUMMARY")
print("="*70)

print("\n✓ Data Cleaning:")
print(f"  - Handled {missing_data['Missing_Count'].sum()} missing values")
print(f"  - Removed Cabin column (77% missing)")
print(f"  - No duplicate rows found")

print("\n✓ Feature Engineering:")
print(f"  - Created Title, FamilySize, IsAlone, AgeGroup, FarePerPerson features")

print("\n✓ Statistical Insights:")
print(f"  - Sex & Survived: Highly significant (χ² test, p < 0.001)")
print(f"  - Pclass & Survived: Highly significant (χ² test, p < 0.001)")
print(f"  - Age & Survived: Significant negative correlation (younger = better survival)")
print(f"  - Fare & Survived: Significant positive correlation (higher fare = better survival)")

print("\n✓ Key Predictors of Survival (in order):")
print(f"  1. Sex (Female > Male)")
print(f"  2. Passenger Class (1st > 2nd > 3rd)")
print(f"  3. Age (Children & Young > Elderly)")
print(f"  4. Fare (Higher = Better survival chances)")
print(f"  5. Family Size (Traveling with family helped)")

print("\n" + "="*70)
print("\nDataset is now clean and ready for machine learning modeling!")